# Logistic Regression

Logistic regression is a statistical method used for binary classification. It is used to predict the probability of an event occurring or not. It's a type of generalized linear model that is used when the dependent variable is binary or categorical. 

In logistic regression, the dependent variable is binary and the independent variables can be either continuous or categorical. The goal of logistic regression is to find the relationship between t he independent variables and the dependent variable by estimating the probability of the dependent variable being 1 given the values of the independent variables. 

$$P(y=1 | x)$$

1. Binomial logistic regression
2. Multinominal logistics regression
3. Ordinal logistic regression

Linear regression is $z = w^Tx + b$,
The logistic regression model uses a logistic function (i.e. sigmoid function $\sigma(z) = \frac{1}{1 + e^{-z}}$) to convert $z$ into a probability between 0 and 1


Instead of modeling probability, logistic regression models log-odds (i.e. odds of an event occurring):
$$\text{log-odds} = log\frac{p}{1 - p}$$

This is log of odds and also called logit, range is $(-\infty, +\infty)$


Hence we can model it as linear:
$$log(\frac{p}{1-p}) = w^Tx + b$$
Solving for $p$, we get sigmoid function, which output is $(0, 1)$:
$$p = \frac{1}{1 + e^{-(w^Tx + b)}} = \frac{1}{1 + e^{-z}}$$

So logistic regression is a **linear model in log-odds space**. 
if features $x_j$ increases by 1:
- log-odds increase by $w_j$
- odds multiply by $e^wj$

**Q: Why sigmoid?**  
It's a smooth probability squasher
- Larger positive $z$ -> probability ~= 1
- Large negative $z$ -> probability ~= 0
- $z$ = 0 -> probability = 0.5
Features combine linearly in log-odds space, then sigmoid converts to probability

So the pipeline is:
$$x \rightarrow \text{linear score } z \rightarrow \text{sigmoid} \rightarrow p$$


## Binary Logistic Regression (Sigmoid)

For binary classification

$$p(y=1 | x) =\sigma(z) = \sigma(w^Tx + b)$$

where sigmoid:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

#### Loss function (Binary Cross Entropy)
$$L = -\frac{1}{n} \sum^n_{i=1}[y_ilog(p_i) + (1-y_i)log(1-p_i)]$$

#### Gradients
let:
$$p = \sigma(Xw + b)$$
Then:
$$\frac{\partial L}{\partial w} = \frac{1}{n} X^T(p-y)$$
$$\frac{\partial L}{\partial b} = \frac{1}{n}\sum{(p-y)}$$


In [25]:
import numpy as np

class LogisticRegression:
    def __init__(self, lr=0.001, max_iter=20):
        self.lr = lr
        self.max_iter = 20
        self.w = None
        self.b = None

    def _sigmoid(self, x):
        return 1/(1 + np.exp(-x))

    def _loss(self, y, p):
        eps = 1e-15
        n = len(y)
        p = np.clip(p, eps, 1-eps)
        return -1/n*np.sum(y*np.log(p) + (1 - y)*np.log(1-p))
    
    def fit(self, X, y):
        n, d = X.shape
        self.w = np.ones(d)
        self.b = 0

        for _ in range(self.max_iter):
            z = X@self.w + self.b
            error = self._sigmoid(z)-y
            dw = 1/n*X.T@error
            db = 1/n*np.sum(error)
            self.w -= self.lr * dw 
            self.b -= self.lr * db

    def predict(self, X, thresh=0.5):
        p = X@self.w + self.b
        return (self._sigmoid(p) >= thresh).astype(int)

    
            

In [26]:
X = np.random.rand(4, 5)
y = np.random.rand(4)

LR = LogisticRegression()
LR.fit(X, y)
LR.predict(X)

array([1, 1, 1, 1])

## Multiclass Logistic regession (Softmax)

Now suppose we have $K$ classes:
$$y \in \{0, 1, 2, ..., K-1\}$$

Binary sigmoid no longer works, so we use softmax instead

Weights is a matrix, and the score is:
$$Z = XW, \quad W\in \mathbb{R}^{d\times K}$$

Softmax:
$$p_{i,k} = \frac{e^{z_{i,k}}}{\sum^K_{j=1}e^{z_{i,j}}}$$
Each row becomes a probability distribution over classes

#### Loss (Cross Entropy)
If $Y$ is one-hot encoded:
$$L = -\frac{1}{n}\sum\sum Y_{i,k}log(p_{i,k})$$


#### Gradient

$$\nabla_W = \frac{1}{n}X^T (P-Y)$$

In [65]:
class MultiLogisticRegression:
    def __init__(self, lr=0.001, max_iter=20):
        self.lr = lr
        self.max_iter = max_iter
        self.W = None
        self.b = None
        # self.K = num_classes

    def _softmax(self, Z):
        Z = Z - np.max(Z, axis=1, keepdims=True)  # stability
        e = np.exp(Z) #n x K
        return e/np.sum(e, axis=1, keepdims=True) 


    def fit(self, X, y):
        n, d = X.shape
        K = len(np.unique(y))
        self.W = np.zeros((d, K))
        self.b = np.zeros(K)

        Y = np.eye(K)[y] #one hot encoding. [y] means indexing
        for _ in range(self.max_iter):
            Z = X@self.W + self.b
            P = self._softmax(Z)
            dw = 1/n * X.T@(P-Y)
            db = 1/n * np.sum(P-Y, axis=0) #K 

            self.W -= self.lr * dw
            self.b -= self.lr * db
    
    def pred(self, X):
        Z = X@self.W + self.b 
        P = self._softmax(Z)
        return np.argmax(P, axis=1) 
    

In [68]:
'''
X: (n, d)
y: (n,)
W: (d, K)
b: (K,)
Z: (n, K)
P: (n, K)
Y: (n, K)

dw: (d, K)
db: (K,)
'''

'\nX: (n, d)\ny: (n,)\nW: (d, K)\nb: (K,)\nZ: (n, K)\nP: (n, K)\nY: (n, K)\n\ndw: (d, K)\ndb: (K,)\n'

In [64]:
X = np.random.rand(4,6)
Y = np.array([0,2,1,2])
LR = MultiLogisticRegression()
LR.fit(X, Y)
LR.pred(X)

array([2, 2, 2, 2])

In [50]:
y = np.array([2,0,2])
np.eye(3)[y]

array([[0., 0., 1.],
       [1., 0., 0.],
       [0., 0., 1.]])